# Introduction to Synthetic Data Generation

This notebook provides a hands-on tour of all synthesis and data preparation methods in this repository.

Each section contains a minimal runnable code example. For the full treatment of each method, follow the link to the dedicated chapter notebook.

**Authors:** Angel Marchev, Vasil Marchev — UNWE Sofia

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 4)

---
## 1. Decision Guide

The decision framework for choosing the right synthesis approach is based on what data is available:

```
What data is available?
│
├─ No data at all
│    └─ Monte Carlo feature-wise + business logic filtering
│
├─ Marginal distributions + correlation matrix known
│    └─ Cholesky decomposition  →  notebooks/ch08_cholesky/
│
├─ Multivariate joint distribution (copula) available
│    └─ Inverse copula sampling  →  notebooks/ch13_copula/
│
├─ Partial real features + business rules
│    └─ Feature derivation + filtering           (Ch. 9)
│
├─ Multiple partial datasets with overlapping keys
│    └─ Probabilistic Concatenation / ProbCon  →  notebooks/ch11_probcon/
│
└─ Full real dataset (confidential / privacy-restricted)
     └─ Generative Adversarial Network (GAN)  →  notebooks/ch12_gan/
```

After generation, all methods pass through:
- **Horizontal synchronisation** — cross-variable logical consistency checks
- **Vertical feature validation** — distributional fidelity (KS test, HECA, LLM-PA)

| Available inputs | Recommended method | Chapter |
|---|---|---|
| Only marginal distributions | Monte Carlo simulation | Ch. 3 |
| Distributions + correlation matrix | Cholesky decomposition | Ch. 8 |
| Multivariate joint distribution | Inverse copula sampling | Ch. 13 |
| Partial real dataset + business rules | Feature engineering + filtering | Ch. 9 |
| Multiple partial datasets with overlap | ProbCon (fuzzy matching) | Ch. 11 |
| Full real dataset (confidential) | GAN | Ch. 12 |

---
## 2. Data Imputation

Missing values must be handled before synthesis. Two strategies are common:

**Mean by category** — replace a missing value with the mean of that variable within its category group. Preserves within-group distributions without distorting group means.

**Noisy input** — replace with a random draw from the empirical distribution of the observed values for that variable. Preserves the overall marginal distribution (no shrinkage toward the mean).

_Full treatment: [`01_missing_values_by_category`](../01_imputation/01_missing_values_by_category.ipynb), [`02_missing_values_noisy_input`](../01_imputation/02_missing_values_noisy_input.ipynb)_

In [ ]:
# Strategy 1: fill missing values with the within-group mean
df = pd.DataFrame({
    'category': ['A', 'A', 'A', 'B', 'B', 'B', 'A', 'B'],
    'income':   [50, np.nan, 60, 80, np.nan, 90, 55, 85]
})

df['income_filled'] = df.groupby('category')['income'].transform(
    lambda x: x.fillna(x.mean())
)
print(df)

In [ ]:
# Strategy 2: fill with a random draw from the observed values (noisy input)
def fill_noisy(series):
    observed = series.dropna().values
    missing_idx = series[series.isna()].index
    series = series.copy()
    series.loc[missing_idx] = np.random.choice(observed, size=len(missing_idx))
    return series

df['income_noisy'] = fill_noisy(df['income'])
print(df[['category', 'income', 'income_noisy']])

---
## 3. Random Naive Oversampling

When a dataset is class-imbalanced, models trained on it will be biased toward the majority class. The simplest remedy is **random oversampling**: draw samples from the minority class with replacement until the classes are balanced.

This preserves the exact observed feature values — it introduces no new information, only replication. It works best when the minority class is well-characterised by the existing samples.

_Full treatment: [`01_random_naive_oversampling`](../02_oversampling/01_random_naive_oversampling.ipynb)_

In [ ]:
# Simulate an imbalanced dataset
majority = pd.DataFrame({'x': np.random.randn(200), 'y': np.random.randn(200), 'label': 0})
minority = pd.DataFrame({'x': np.random.randn(20) + 2, 'y': np.random.randn(20) + 2, 'label': 1})
df_imbalanced = pd.concat([majority, minority], ignore_index=True)

print("Before:", df_imbalanced['label'].value_counts().to_dict())

n_majority = (df_imbalanced['label'] == 0).sum()
minority_upsampled = (
    df_imbalanced[df_imbalanced['label'] == 1]
    .sample(n=n_majority, replace=True, random_state=42)
)
df_balanced = pd.concat(
    [df_imbalanced[df_imbalanced['label'] == 0], minority_upsampled],
    ignore_index=True
).sample(frac=1, random_state=42)

print("After: ", df_balanced['label'].value_counts().to_dict())

---
## 4. SMOTE

**Synthetic Minority Over-sampling TEchnique (SMOTE)** creates *new* synthetic minority samples rather than replicating existing ones. For each minority sample $x_i$, it:
1. Finds its $k$ nearest neighbours in the minority class
2. Selects a random neighbour $x_j$
3. Creates a new sample at $x_i + \lambda (x_j - x_i)$ where $\lambda \sim \text{Uniform}(0, 1)$

This interpolates between real minority samples, generating plausible new observations rather than exact duplicates.

_Full treatment: [`02_smote`](../02_oversampling/02_smote.ipynb)_

In [ ]:
from imblearn.over_sampling import SMOTE

X = df_imbalanced[['x', 'y']].values
y = df_imbalanced['label'].values

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

print("Before SMOTE:", dict(zip(*np.unique(y, return_counts=True))))
print("After SMOTE: ", dict(zip(*np.unique(y_res, return_counts=True))))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (data, labels, title) in zip(axes, [
    (X, y, 'Original (imbalanced)'),
    (X_res, y_res, 'After SMOTE')
]):
    for label, colour in [(0, 'steelblue'), (1, 'orange')]:
        mask = labels == label
        ax.scatter(data[mask, 0], data[mask, 1], c=colour, alpha=0.4, s=15, label=f'class {label}')
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Monte Carlo Simulation

Monte Carlo simulation generates synthetic observations by sampling from a known or fitted probability distribution many times. It is the method of choice when no real data exists — only domain knowledge about the marginal distributions of each feature.

The classic demonstration is the dice roll: simulate many rolls to recover the theoretical probability distribution of the sum.

_Full treatment: [`notebooks/03_monte_carlo/01_dice_distribution.ipynb`](../03_monte_carlo/01_dice_distribution.ipynb)_

In [ ]:
n_trials = 50_000
n_dice, n_sides = 2, 6

rolls = np.random.randint(1, n_sides + 1, size=(n_trials, n_dice))
sums = rolls.sum(axis=1)

# Empirical vs theoretical distribution
values, counts = np.unique(sums, return_counts=True)
empirical_prob = counts / n_trials

# Exact probability for 2d6
theoretical = {s: (min(s - 1, 13 - s)) / 36 for s in range(2, 13)}

plt.bar(values - 0.2, empirical_prob, width=0.4, label='Monte Carlo', alpha=0.8)
plt.bar(values + 0.2, [theoretical[v] for v in values], width=0.4, label='Theoretical', alpha=0.8)
plt.xlabel('Sum of two dice')
plt.ylabel('Probability')
plt.title(f'Monte Carlo ({n_trials:,} trials) vs Exact distribution')
plt.legend()
plt.tight_layout()
plt.show()

---
## 6. Distribution Fitting

Before applying Cholesky or Monte Carlo synthesis to real variables, we need to identify which theoretical distribution best describes each feature. The `fitter` library fits all common distributions from `scipy.stats` and ranks them by goodness-of-fit.

_Full treatment: [`notebooks/03_monte_carlo/02_distribution_fitting.ipynb`](../03_monte_carlo/02_distribution_fitting.ipynb)_

In [ ]:
from fitter import Fitter

# Generate sample data with a known distribution (log-normal, typical for income)
sample_data = np.random.lognormal(mean=10, sigma=0.5, size=2000)

f = Fitter(sample_data, distributions=['norm', 'lognorm', 'gamma', 'expon', 'beta'])
f.fit()
print(f.summary())

print("\nBest fit:", f.get_best())

---
## 7. Cholesky Decomposition

**When to use:** marginal distributions and a correlation (or covariance) matrix are known or can be estimated from a sample.

Cholesky decomposition factorises a positive-definite covariance matrix $\Sigma = L L^\top$ where $L$ is lower triangular. Multiplying independent standard normal samples $Z$ by $L^\top$ produces correlated samples with the target covariance:

$$X = Z \cdot L^\top$$

After generating correlated standard normal variables, apply the inverse CDF of each feature's fitted marginal distribution to map to the correct scale.

_Full treatment: [`notebooks/ch08_cholesky/cholesky_decomposition.ipynb`](../ch08_cholesky/cholesky_decomposition.ipynb)_

In [ ]:
def generate_correlated(mean, cov, n):
    """Generate n samples from a multivariate distribution using Cholesky decomposition."""
    L = np.linalg.cholesky(cov)
    z = np.random.randn(n, len(mean))
    return mean + z @ L.T

# Two correlated financial variables: income and expenditure (correlation = 0.75)
mean = np.array([5000, 3000])
cov  = np.array([[400_000, 150_000],
                 [150_000,  90_000]])

synth = generate_correlated(mean, cov, n=2000)
print(f"Synthetic means:   {synth.mean(axis=0).round(1)}")
print(f"Target means:      {mean}")
print(f"Synthetic corr:    {np.corrcoef(synth.T)[0,1]:.3f}")
print(f"Target corr:       {cov[0,1] / np.sqrt(cov[0,0]*cov[1,1]):.3f}")

plt.scatter(synth[:, 0], synth[:, 1], alpha=0.3, s=8)
plt.xlabel('Synthetic income')
plt.ylabel('Synthetic expenditure')
plt.title('Cholesky-generated correlated variables (r = 0.75)')
plt.tight_layout()
plt.show()

For N variables, define the full correlation matrix, check it is positive definite, and apply the same factorisation. The full pipeline in `ch08_cholesky` also fits marginal distributions with `fitter` and validates output with the Kolmogorov–Smirnov test.

In [ ]:
def is_positive_definite(matrix):
    try:
        np.linalg.cholesky(matrix)
        return True
    except np.linalg.LinAlgError:
        return False

# 4-variable example
corr = np.array([
    [1.00, 0.75, 0.50, 0.30],
    [0.75, 1.00, 0.40, 0.20],
    [0.50, 0.40, 1.00, 0.60],
    [0.30, 0.20, 0.60, 1.00]
])
print("Positive definite:", is_positive_definite(corr))

L = np.linalg.cholesky(corr)
z = np.random.randn(5000, 4)
X_synth = z @ L.T

print("\nEmpirical correlation matrix:")
print(np.corrcoef(X_synth.T).round(2))

---
## 8. Gaussian Copula

**When to use:** the full multivariate joint distribution is available (or can be estimated), not just marginals + correlations.

A Gaussian copula separates the dependence structure from the marginal distributions:
1. Transform each variable to a uniform marginal $U[0,1]$ via its empirical CDF
2. Apply the normal quantile function to get standard normal marginals
3. Use Cholesky decomposition to introduce the target correlations
4. Map back to each variable's original scale via the inverse CDF

This allows synthesising correlated data even when marginals are non-normal (e.g., categorical, skewed, bounded).

_Full treatment: [`notebooks/ch13_copula/cold_modeling_copula.ipynb`](../ch13_copula/cold_modeling_copula.ipynb)_

In [ ]:
from scipy.stats import norm

def gaussian_copula(n_samples, corr_matrix, marginal_rvs):
    """
    Generate samples from a Gaussian copula.
    marginal_rvs: list of callables, each takes n and returns samples from one marginal.
    """
    n_vars = len(marginal_rvs)
    L = np.linalg.cholesky(corr_matrix)

    # Correlated standard normals
    z = np.random.randn(n_samples, n_vars) @ L.T

    # Map to uniform via the normal CDF
    u = norm.cdf(z)

    # Map each uniform margin to the target marginal via inverse CDF (percent-point function)
    samples = np.column_stack([rv(u[:, i]) for i, rv in enumerate(marginal_rvs)])
    return samples


# Example: two correlated variables with different marginals
# Variable 1: log-normal (income-like), Variable 2: normal (expenditure-like)
corr = np.array([[1.0, 0.65], [0.65, 1.0]])

marginals = [
    lambda u: stats.lognorm.ppf(u, s=0.5, scale=np.exp(10)),  # log-normal
    lambda u: stats.norm.ppf(u, loc=3000, scale=300),          # normal
]

data = gaussian_copula(n_samples=3000, corr_matrix=corr, marginal_rvs=marginals)

print(f"Empirical correlation: {np.corrcoef(data.T)[0,1]:.3f}  (target: 0.65)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(data[:, 0], data[:, 1], alpha=0.2, s=8)
axes[0].set_xlabel('Log-normal variable')
axes[0].set_ylabel('Normal variable')
axes[0].set_title('Joint distribution (Gaussian copula)')
axes[1].hist(data[:, 0], bins=40, edgecolor='none', alpha=0.8)
axes[1].set_title('Marginal: log-normal')
plt.tight_layout()
plt.show()

---
## 9. GAN

**When to use:** a real confidential or privacy-restricted dataset exists and we want a synthetic surrogate that closely replicates its full statistical structure without exposing individual records.

A Generative Adversarial Network trains two competing networks:
- **Generator** $G$: maps random noise $z$ to synthetic samples
- **Discriminator** $D$: distinguishes real from synthetic samples

Training alternates: $D$ learns to detect fakes; $G$ learns to fool $D$. At convergence, $G(z)$ produces samples statistically indistinguishable from the real data.

Pre-trained models are stored in `models/`. Validation uses cosine similarity of mean feature vectors and k-means cluster analysis.

_Full treatment: [`notebooks/ch12_gan/gan_synthetic_data.ipynb`](../ch12_gan/gan_synthetic_data.ipynb)_

In [ ]:
import tensorflow as tf

def build_generator(noise_dim, output_dim):
    return tf.keras.Sequential([
        tf.keras.layers.Dense(128, activation='relu', input_shape=(noise_dim,)),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(output_dim, activation='linear')
    ], name='generator')

def build_discriminator(input_dim):
    return tf.keras.Sequential([
        tf.keras.layers.Dense(256, activation='relu', input_shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ], name='discriminator')

NOISE_DIM = 100
N_FEATURES = 48  # Bulgarian financial/demographic survey dataset

generator     = build_generator(NOISE_DIM, N_FEATURES)
discriminator = build_discriminator(N_FEATURES)

print(generator.summary())

# To generate from a pre-trained model:
# generator = tf.keras.models.load_model('../../models/generator.tf')
# noise = tf.random.normal([1000, NOISE_DIM])
# synthetic = generator(noise, training=False).numpy()

The training loop applies:
- **Discriminator loss:** binary cross-entropy — real samples labelled 1, synthetic labelled 0
- **Generator loss:** binary cross-entropy — synthetic samples labelled 1 (fool the discriminator)

```python
cross_entropy = tf.keras.losses.BinaryCrossentropy()

def discriminator_loss(real_output, fake_output):
    return cross_entropy(tf.ones_like(real_output), real_output) + \
           cross_entropy(tf.zeros_like(fake_output), fake_output)

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)
```

See the full GAN notebook for the complete training loop, early stopping, and validation with the KS test.

---
## 10. ProbCon

**When to use:** multiple partial datasets from different sources contain overlapping key variables (e.g., name + city) but values are not identical across sources due to spelling variation, abbreviation, or data entry errors.

**Probabilistic Concatenation (ProbCon)** merges these datasets by:
1. Tokenising key fields and building a TF-IDF candidate index (SQLite)
2. Computing match probabilities between candidate pairs (Levenshtein distance + TF-IDF score)
3. Resolving ambiguous matches stochastically by sampling from the probability distribution of candidates

The result is a probabilistically joined dataset — each record in the left table is matched to the most probable record in the right table, with match confidence scores for validation.

_Full treatment: [`notebooks/ch11_probcon/01_probabilistic_concatenation.ipynb`](../ch11_probcon/01_probabilistic_concatenation.ipynb)_

In [ ]:
import sys
sys.path.insert(0, '../../')
import fuzzymatcher

# Survey dataset 1: name + city + income
df_survey = pd.DataFrame({
    'id_survey': [1, 2, 3, 4],
    'name_survey': ['Ivan Petrov', 'Maria Georgieva', 'Georgi Ivanov', 'Elena Dimitrova'],
    'income': [2500, 3100, 1800, 4200]
})

# Registry dataset: same people, different source (slight name variation)
df_registry = pd.DataFrame({
    'id_registry': ['A', 'B', 'C', 'D'],
    'name_registry': ['I. Petrov', 'M. Georgieva', 'G. Ivanov', 'E. Dimitrova'],
    'education': ['MSc', 'BSc', 'PhD', 'MSc']
})

# Fuzzy left join on the name columns
result = fuzzymatcher.fuzzy_left_join(
    df_survey, df_registry,
    left_on=['name_survey'],
    right_on=['name_registry']
)

print(result[['name_survey', 'name_registry', 'best_match_score', 'income', 'education']])

---
## Summary

| Method | Key library | Typical output size | Chapter |
|---|---|---|---|
| Imputation (mean by category) | `pandas` | — (in-place) | Ch. 2.2 |
| Imputation (noisy input) | `numpy` | — (in-place) | Ch. 2.2 |
| Random naive oversampling | `pandas` | up to 2× minority | Ch. 3.3 |
| SMOTE | `imbalanced-learn` | up to 2× minority | Ch. 3.3 |
| Monte Carlo | `numpy` | unlimited | Ch. 6.3.3 |
| Cholesky decomposition | `numpy`, `scipy` | unlimited | Ch. 8 |
| Gaussian copula | `scipy`, `numpy` | unlimited | Ch. 13 |
| GAN | `tensorflow` | unlimited | Ch. 12 |
| ProbCon | `fuzzymatcher` | ≤ left table size | Ch. 11 |

Each chapter notebook includes full pipelines, validation (KS test, cluster analysis, cosine similarity), and Bulgarian financial/demographic survey data where applicable.